In [207]:
import pandas as pd
from statsforecast.models import MSTL
from statsforecast.models import MSTL
import pandas as pd
from google.cloud import bigquery
import datetime
import json

In [3]:
client = bigquery.Client()

/mnt/encrypted_data/git/data-testing/venv/lib/python3.9/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/mnt/encrypted_data/git/data-testing/venv/lib/python3.9/site-packages/google/auth/_default.py:78: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [186]:
EXECUTION_DATE = "2023-01-01"
current_forecast_config = {
    "source_dataset": "pipe_ais_v3_alpha_published",
    "source_table": "stats_daily",
    "source_date_column": "date",
    "source_date_column_sql": "date",
    "source_forecast_column": "raw_positions",
    "source_forecast_column_sql": "MAX(raw_positions)",
    "source_sql": None,

    "algorithms": {
        "mstl": {
            "parameters": {"season_length": [365, 7]},
            "train_start": "2012-01-01",
            "train_end": EXECUTION_DATE,
            "forecast_periods": 3
        }
    },

    "target_dataset": "scratch_christian_homberg_ttl120d",
    "target_table": "anomaly_detection_forecasts"
}

In [187]:
for key, value in current_forecast_config.items():
    globals()[key] = value

In [188]:
def get_training_data(train_start=None, train_end=None):
    if (source_sql is not None):
        df_timeseries = pd.read_gbq(source_sql)
    else:
        df_timeseries = pd.read_gbq(f'''
        SELECT 
            {source_date_column_sql} date, 
            {source_forecast_column_sql} y
        FROM {source_dataset}.{source_table}
        WHERE {source_date_column_sql} BETWEEN '{train_start}' AND '{train_end}'
        GROUP BY date
        ORDER BY date
        ''')

    return(df_timeseries["y"].to_numpy().astype(int))

In [189]:
current_forecast_config["forecasts"] = {}

In [312]:
if "mstl" in algorithms:
    mstl_config = algorithms["mstl"]
    np_mstl_train=get_training_data(mstl_config["train_start"], mstl_config["train_end"])
    mstl_model = MSTL(**mstl_config["parameters"])
    mstl_forecasts = mstl_model.forecast(np_mstl_train, mstl_config["forecast_periods"])["mean"]
    mstl_train_end = datetime.date.fromisoformat(mstl_config["train_end"])
    mstl_forecast_dates = [(mstl_train_end + datetime.timedelta(days=d)) for d in range(1, mstl_config["forecast_periods"] + 1)]
    current_forecast_config["forecast_algorithm"] = "mstl"
    current_forecast_config["forecasts"] = [{"date": k, "value": v} for k,v in zip(mstl_forecast_dates, mstl_forecasts)]
    

In [329]:
df_forecasts = pd.DataFrame.from_dict([current_forecast_config]) \
    .explode("forecasts") \
    .assign(forecast_date = lambda row: row["forecasts"].str["date"]) \
    .assign(forecast_value = lambda row: row["forecasts"].str["value"]) \
    .drop(["forecasts"], axis=1) \
    .assign(execution_time = datetime.datetime.now())

df_forecasts = df_forecasts.astype({"algorithms": "string"})

In [330]:
table_schema = [
    {"name": "source_dataset", "type": "STRING"},
    {"name": "source_table", "type": "STRING"},
    {"name": "source_date_column", "type": "STRING"},
    {"name": "source_date_column_sql", "type": "STRING"},
    {"name": "source_forecast_column", "type": "STRING"},
    {"name": "source_forecast_column_sql", "type": "STRING"},
    {"name": "source_sql", "type": "STRING"},
    {"name": "algorithms", "type": "STRING"},
    {"name": "target_dataset", "type": "STRING"},
    {"name": "target_table", "type": "STRING"},
    {"name": "algorithm", "type": "STRING"},
    {"name": "forecast_algorithm", "type": "STRING"},
    {"name": "forecast_date", "type": "DATE"},
    {"name": "forecast_value", "type": "FLOAT"},
    {"name": "execution_time", "type": "DATETIME"}
]


df_forecasts.to_gbq(destination_table=f"{target_dataset}.{target_table}", if_exists="replace", table_schema=table_schema)

100%|██████████| 1/1 [00:00<00:00, 16008.79it/s]
